# 05d — Clustering Hierarchical (Ward)

Linkage sobre muestra de 25k. KMeans con centroides para proyectar al resto.

In [1]:
import pandas as pd
import numpy as np
import time, joblib
from pathlib import Path
from datetime import datetime
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder

ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase4_gustos' else Path.cwd()
DATA_QC = ROOT / 'data' / 'data_qc_gustos'
DATA_MODELS = ROOT / 'data' / 'data_models_gustos'
INFORMES_BASE = ROOT / 'informes' / 'fase4_gustos' / '04_clustering'
DATA_MODELS.mkdir(parents=True, exist_ok=True)

MASTER_VERSIONS = {
    'v1_conservative': DATA_QC / 'master_table_gustos_v1_conservative.parquet',
    'v2_intermediate': DATA_QC / 'master_table_gustos_v2_intermediate.parquet',
    'v3_aggressive':   DATA_QC / 'master_table_gustos_v3_aggressive.parquet',
}

def reduce_high_card(df, max_unique=10):
    out = df.copy()
    for cat in out.select_dtypes(include=['object', 'category']).columns.tolist():
        n = out[cat].nunique(dropna=True)
        if n > max_unique:
            top = out[cat].value_counts().head(max_unique).index.tolist()
            out[cat] = out[cat].where(out[cat].isin(top), 'OTHER')
    return out

def preprocess_master(master_df, nan_threshold_zero=0.30):
    """Returns (X_scaled, preprocessor, feature_names_out)."""
    df = master_df.drop(columns=['user_id']).copy()
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    categorical_cols = df.select_dtypes(include=['object','category']).columns.tolist()
    nan_pcts = df[numeric_cols].isna().mean()
    num_low = nan_pcts[nan_pcts < nan_threshold_zero].index.tolist()
    num_high = nan_pcts[nan_pcts >= nan_threshold_zero].index.tolist()
    print(f'  num_low_nan: {len(num_low)} | num_high_nan: {len(num_high)} | cat: {len(categorical_cols)}')
    df = reduce_high_card(df, max_unique=10)
    preproc = ColumnTransformer([
        ('num_low', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', RobustScaler())]), num_low),
        ('num_high', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value=0)), ('sc', RobustScaler())]), num_high),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='missing')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical_cols),
    ])
    X = preproc.fit_transform(df)
    feature_names = preproc.get_feature_names_out()
    return X, preproc, feature_names

def load_and_preprocess(version_name):
    master = pd.read_parquet(MASTER_VERSIONS[version_name])
    print(f'[{version_name}] shape={master.shape}')
    user_ids = master['user_id'].values
    X, preproc, names = preprocess_master(master)
    return X, user_ids, preproc, names, master.shape

from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

REPORT = INFORMES_BASE / '05d_hierarchical'
REPORT.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
K_RANGE = list(range(2, 11))
SAMPLE_HIERARCHICAL = 25000
SAMPLE_FOR_SILHOUETTE = 20000
results = []


In [2]:

for version in MASTER_VERSIONS:
    print(f'\n=== {version} ===')
    X, user_ids, preproc, names, shape = load_and_preprocess(version)
    print(f'  X shape: {X.shape}')

    rng = np.random.RandomState(RANDOM_STATE)
    if len(X) > SAMPLE_HIERARCHICAL:
        sample_idx = rng.choice(len(X), SAMPLE_HIERARCHICAL, replace=False)
        X_sample = X[sample_idx]
    else:
        sample_idx = np.arange(len(X))
        X_sample = X

    t0 = time.time()
    print('  Computing linkage (ward)...')
    Z = linkage(X_sample, method='ward')
    print(f'  linkage in {time.time()-t0:.1f}s')

    # Dendrograma
    fig, ax = plt.subplots(figsize=(12, 4))
    dendrogram(Z, truncate_mode='level', p=5, ax=ax, no_labels=True)
    ax.set_title(f'Dendrograma (sample 25k) — {version}')
    ax.set_xlabel('Cluster (truncated)')
    plt.tight_layout()
    plt.savefig(REPORT / f'dendrogram_{version}.png', dpi=120, bbox_inches='tight')
    plt.close()

    sil_idx = rng.choice(len(X), min(SAMPLE_FOR_SILHOUETTE, len(X)), replace=False)
    X_sil = X[sil_idx]

    for k in K_RANGE:
        t0 = time.time()
        labels_sample = fcluster(Z, t=k, criterion='maxclust')
        # Centroides de la muestra
        centroids = np.array([X_sample[labels_sample == i].mean(axis=0) for i in range(1, k+1)])
        # Proyectar al resto via 1-iter KMeans
        km = KMeans(n_clusters=k, init=centroids, n_init=1, random_state=RANDOM_STATE)
        labels_all = km.fit_predict(X)
        labels_sub = km.predict(X_sil)
        try:
            sil = float(silhouette_score(X_sil, labels_sub)) if len(set(labels_sub)) > 1 else float('nan')
        except Exception:
            sil = float('nan')
        try:
            db = float(davies_bouldin_score(X, labels_all))
        except Exception:
            db = float('nan')
        elapsed = time.time() - t0
        results.append({
            'algorithm': 'Hierarchical', 'version': version, 'k': k,
            'silhouette': sil, 'davies_bouldin': db,
            'n_clusters_actual': int(len(set(labels_all))),
            'elapsed_s': elapsed,
        })
        print(f'  K={k}: sil={sil:.4f} db={db:.4f} t={elapsed:.1f}s')
        joblib.dump({'model': km, 'Z': Z, 'labels': labels_all, 'user_ids': user_ids, 'sample_idx': sample_idx}, DATA_MODELS / f'hierarchical_{version}_k{k}.joblib')

res_df = pd.DataFrame(results)
res_df.to_csv(REPORT / 'metrics.csv', index=False)
best = res_df.sort_values('silhouette', ascending=False).groupby('version').head(1)
lines = [f'# 05d Hierarchical', f'**Fecha**: {datetime.now().strftime("%Y-%m-%d %H:%M")}', '',
         '## K óptimo por max silhouette', '', '| Versión | K | silhouette | Davies-Bouldin |', '|---|---:|---:|---:|']
for _, r in best.iterrows():
    lines.append(f"| {r['version']} | {int(r['k'])} | {r['silhouette']:.4f} | {r['davies_bouldin']:.4f} |")
(REPORT / 'execution_report.md').write_text('\n'.join(lines))



=== v1_conservative ===
[v1_conservative] shape=(114412, 115)


/var/folders/0f/3zq36dr1451_ttvt93n947ph0000gw/T/ipykernel_4205/1977623806.py:36: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object','category']).columns.tolist()
/var/folders/0f/3zq36dr1451_ttvt93n947ph0000gw/T/ipykernel_4205/1977623806.py:25: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence t

  num_low_nan: 86 | num_high_nan: 24 | cat: 4


  X shape: (114412, 132)
  Computing linkage (ward)...


  linkage in 15.9s


...

  K=7: sil=0.9872 db=0.3879 t=2.1s


  K=8: sil=0.9871 db=0.4033 t=2.1s


  K=9: sil=0.9756 db=0.4181 t=2.2s


  K=10: sil=0.9641 db=0.4406 t=2.1s


270